# Job: Ingestão para a Landing

1. Baixa os dados brutos da MUNIC da API do IBGE
2. Salva no schema landing no formato .xlsx (a forma original)

In [0]:
%run ../environment/env_variables

In [0]:
import requests
import hashlib
import os

In [0]:
# print(f"Origem : {munic_source_url}")
# print(f"Destino: {munic_file}")


In [0]:
def download_file(munic_file, munic_source_url):
    # Verifica se o arquivo já existe
    if os.path.exists(munic_file):
        print(f"Arquivo já existe em {munic_file} — download ignorado.")
        return False
    else:
        # Faz a requisição à API do IBGE
        file = requests.get(munic_source_url)

        # Lança uma exceção caso ocorra um erro na requisição
        file.raise_for_status()

        # Grava o conteúdo da resposta no arquivo xlsx
        with open(munic_file, "wb") as f:
            f.write(file.content)
        
        print(f"Arquivo baixado com sucesso em {munic_file}")
        return True


def get_file_sha256(munic_file, munic_file_name):
    with open(munic_file, "rb") as f:
        file_bytes = f.read()

    file_sha256 = hashlib.sha256(file_bytes).hexdigest()

    # Exibe as informações do arquivo

    print(f"Arquivo : {munic_file_name}")
    print(f"Tamanho : {len(file_bytes):,} bytes")
    print(f"SHA-256 : {file_sha256}\n")

In [0]:
# Para cada edição registrada em munic_source_url_map, verifica se o arquivo já existe na landing
# Caso não exista, baixa do IBGE e salva no caminho definido em munic_file_paths
for year, url in munic_source_url_map.items():
    file_path = munic_file_paths[year]
    file_name = url.split("/")[-1]
    download_file(file_path, url)

    # Calcula e exibe o SHA-256 sempre, verificando a integridade mesmo de arquivos já existentes
    get_file_sha256(file_path, file_name)